In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


In [9]:
with open('/content/drive/MyDrive/Computer Vision/utils/__init__.py', 'w') as f:
    f.write("# This file marks 'utils' as a Python package.\n")


In [10]:


import sys
sys.path.append('/content/drive/MyDrive/Computer Vision')


In [ ]:
# library
# standard library
import os, sys

# third-party library
import numpy as np
import collections
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import DataLoader
from dataset import dataset_processing
from timeit import default_timer as timer
from utils.report import report_precision_se_sp_yi, report_mae_mse
from utils.utils import Logger, AverageMeter, time_to_str, weights_init
from utils.genLD import genLD
from model.resnet50 import resnet50
import torch.backends.cudnn as cudnn
from transforms.affine_transforms import *
import time
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report


# Hyper Parameters
BATCH_SIZE = 32
BATCH_SIZE_TEST = 20
LR = 0.001              # learning rate
NUM_WORKERS = 12
NUM_CLASSES = 4
LOG_FILE_NAME = './logs/log_' + time.strftime("%Y-%m-%d_%H:%M:%S", time.localtime()) + '.log'
lr_steps = [30, 60, 90, 120]

np.random.seed(42)

DATA_PATH = '/content/drive/MyDrive/Classification/JPEGImages'



LOG_FILE_NAME = './logs/log_' + time.strftime("%Y-%m-%d_%H:%M:%S", time.localtime()) + '.log'
lr_steps = [30, 60, 90, 120]

np.random.seed(42)

DATA_PATH = '/content/drive/MyDrive/Classification/JPEGImages'

# Create logs directory if it doesn't exist
os.makedirs('./logs', exist_ok=True)

log = Logger()
log.open(LOG_FILE_NAME, mode="a")




def criterion(lesions_num):
    if lesions_num <= 5:
        return 0
    elif lesions_num <= 20:
        return 1
    elif lesions_num <= 50:
        return 2
    else:
        return 3

def trainval_test(cross_val_index, sigma, lam):

    TRAIN_FILE = '/content/drive/MyDrive/Classification/NNEW_trainval_' + cross_val_index + '.txt'
    TEST_FILE = '/content/drive/MyDrive/Classification/NNEW_test_' + cross_val_index + '.txt'

    normalize = transforms.Normalize(mean=[0.45815152, 0.361242, 0.29348266],
                                     std=[0.2814769, 0.226306, 0.20132513])

    dset_train = dataset_processing.DatasetProcessing(
        DATA_PATH, TRAIN_FILE, transform=transforms.Compose([
                transforms.Resize((256, 256)),
                transforms.RandomCrop(224),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                RandomRotate(rotation_range=20),
                normalize,
            ]))

    dset_test = dataset_processing.DatasetProcessing(
        DATA_PATH, TEST_FILE, transform=transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                normalize,
            ]))

    train_loader = DataLoader(dset_train,
                              batch_size=BATCH_SIZE,
                              shuffle=True,
                              num_workers=NUM_WORKERS,
                              pin_memory=True)

    test_loader = DataLoader(dset_test,
                             batch_size=BATCH_SIZE_TEST,
                             shuffle=False,
                             num_workers=NUM_WORKERS,
                             pin_memory=True)

    cnn = resnet50().cuda()
    cudnn.benchmark = True

    params = []
    new_param_names = ['fc', 'counting']
    for key, value in dict(cnn.named_parameters()).items():
        if value.requires_grad:
            if any(i in key for i in new_param_names):
                params += [{'params': [value], 'lr': LR * 1.0, 'weight_decay': 5e-4}]
            else:
                params += [{'params': [value], 'lr': LR * 1.0, 'weight_decay': 5e-4}]

    optimizer = torch.optim.SGD(params, momentum=0.9)  #

    loss_func = nn.CrossEntropyLoss().cuda()
    kl_loss_1 = nn.KLDivLoss().cuda()
    kl_loss_2 = nn.KLDivLoss().cuda()
    kl_loss_3 = nn.KLDivLoss().cuda()

    def adjust_learning_rate_new(optimizer, decay=0.5):
        """Sets the learning rate to the initial LR decayed by 0.5 every 20 epochs"""
        for param_group in optimizer.param_groups:
            param_group['lr'] = decay * param_group['lr']

    # training and testing
    start = timer()
    test_acc_his = 0.7
    test_mae_his = 8
    test_mse_his = 18

    for epoch in range(lr_steps[-1]):


        if epoch in lr_steps:
            adjust_learning_rate_new(optimizer, 0.5)
        # scheduler.step(epoch)

        losses_cls = AverageMeter()
        losses_cou = AverageMeter()
        losses_cou2cls = AverageMeter()
        losses = AverageMeter()
        # '''
        cnn.train()
        for step, (b_x, b_y, b_l) in enumerate(train_loader):   # gives batch data, normalize x when iterate train_loader

            b_x = b_x.cuda()
            b_l = b_l.numpy()

            # generating ld
            b_l = b_l - 1
            ld = genLD(b_l, sigma, 'klloss', 65)
            ld_4 = np.vstack((np.sum(ld[:, :5], 1), np.sum(ld[:, 5:20], 1), np.sum(ld[:, 20:50], 1), np.sum(ld[:, 50:], 1))).transpose()
            ld = torch.from_numpy(ld).cuda().float()
            ld_4 = torch.from_numpy(ld_4).cuda().float()

            # train
            cnn.train()

            cls, cou, cou2cls = cnn(b_x, None)  # nn output
            loss_cls = kl_loss_1(torch.log(cls), ld_4) * 4.0
            loss_cou = kl_loss_2(torch.log(cou), ld) * 65.0
            loss_cls_cou = kl_loss_3(torch.log(cou2cls), ld_4) * 4.0
            loss = (loss_cls + loss_cls_cou) * 0.5 * lam + loss_cou * (1.0 - lam)
            optimizer.zero_grad()           # clear gradients for this training step
            loss.backward()                 # backpropagation, compute gradients
            optimizer.step()                # apply gradients

            losses_cls.update(loss_cls.item(), b_x.size(0))
            losses_cou.update(loss_cou.item(), b_x.size(0))
            losses_cou2cls.update(loss_cls_cou.item(), b_x.size(0))
            losses.update(loss.item(), b_x.size(0))
        message = '%s %6.0f | %0.3f | %0.3f | %0.3f | %0.3f | %s\n' % ( \
                "train", epoch,
                losses_cls.avg,
                losses_cou.avg,
                losses_cou2cls.avg,
                losses.avg,
                time_to_str((timer() - start), 'min'))
        # print(message)
        log.write(message)
        # '''
       if epoch >= 9:
            epoch_start = timer()
            with torch.no_grad():
                test_loss = 0
                test_corrects = 0
                test_corrects_m = 0  # For merged predictions
                y_true = np.array([])
                y_pred = np.array([])
                y_pred_m = np.array([])
                l_true = np.array([])
                l_pred = np.array([])

                cnn.eval()
                for step, (test_x, test_y, test_l) in enumerate(test_loader):
                    test_x = test_x.cuda()
                    test_y = test_y.cuda()

                    y_true = np.hstack((y_true, test_y.data.cpu().numpy()))
                    l_true = np.hstack((l_true, test_l.data.cpu().numpy()))

                    cls, cou, cou2cls = cnn(test_x, None)
                    loss = loss_func(cou2cls, test_y)
                    test_loss += loss.data

                    _, preds_m = torch.max(cls + cou2cls, 1)
                    _, preds = torch.max(cls, 1)

                    y_pred = np.hstack((y_pred, preds.data.cpu().numpy()))
                    y_pred_m = np.hstack((y_pred_m, preds_m.data.cpu().numpy()))

                    _, preds_l = torch.max(cou, 1)
                    preds_l = (preds_l + 1).data.cpu().numpy()
                    l_pred = np.hstack((l_pred, preds_l))

                    test_corrects += torch.sum((preds == test_y)).data.cpu().numpy()
                    test_corrects_m += torch.sum((preds_m == test_y)).data.cpu().numpy()

                test_loss = test_loss.float() / len(test_loader)
                test_acc = test_corrects / len(test_loader.dataset)
                test_acc_m = test_corrects_m / len(test_loader.dataset)

                # Calculate metrics
                def format_confusion_matrix(cm, class_names=None):
                    if class_names is None:
                        class_names = [str(i) for i in range(NUM_CLASSES)]
                    header = " " * 5 + "  ".join([f"{name:^5}" for name in class_names])
                    rows = []
                    for i, row in enumerate(cm):
                        rows.append(f"{class_names[i]:>5} " + " ".join([f"{x:5}" for x in row]))
                    return "Confusion Matrix:\n" + header + "\n" + "\n".join(rows)

                def calculate_metrics(y_true, y_pred):
                    return {
                        'report': classification_report(y_true, y_pred, digits=4),
                        'cm': confusion_matrix(y_true, y_pred),
                        'precision': precision_score(y_true, y_pred, average='weighted'),
                        'recall': recall_score(y_true, y_pred, average='weighted'),
                        'f1': f1_score(y_true, y_pred, average='weighted'),
                        'precision_per_class': precision_score(y_true, y_pred, average=None),
                        'recall_per_class': recall_score(y_true, y_pred, average=None),
                        'f1_per_class': f1_score(y_true, y_pred, average=None)
                    }

                # Calculate for both prediction types
                metrics_std = calculate_metrics(y_true, y_pred)
                metrics_merged = calculate_metrics(y_true, y_pred_m)

                # Log results
                log.write("\n=== Evaluation Metrics ===\n")
                log.write(f"Test Loss: {test_loss.item():.4f}\n")

                log.write("\nStandard Predictions:\n")
                log.write(f"Accuracy: {test_acc:.4f}\n")
                log.write(f"Precision (weighted): {metrics_std['precision']:.4f}\n")
                log.write(f"Recall (weighted): {metrics_std['recall']:.4f}\n")
                log.write(f"F1 (weighted): {metrics_std['f1']:.4f}\n")
                log.write("Class-wise Precision: " + " ".join([f"{x:.4f}" for x in metrics_std['precision_per_class']]) + "\n")
                log.write("Class-wise Recall:    " + " ".join([f"{x:.4f}" for x in metrics_std['recall_per_class']]) + "\n")
                log.write("Class-wise F1:        " + " ".join([f"{x:.4f}" for x in metrics_std['f1_per_class']]) + "\n")
                log.write(format_confusion_matrix(metrics_std['cm']) + "\n")
                log.write("\nClassification Report:\n" + metrics_std['report'] + "\n")

                log.write("\nMerged Predictions (cls + cou2cls):\n")
                log.write(f"Accuracy: {test_acc_m:.4f}\n")
                log.write(f"Precision (weighted): {metrics_merged['precision']:.4f}\n")
                log.write(f"Recall (weighted): {metrics_merged['recall']:.4f}\n")
                log.write(f"F1 (weighted): {metrics_merged['f1']:.4f}\n")
                log.write(format_confusion_matrix(metrics_merged['cm']) + "\n")
                log.write("\nClassification Report:\n" + metrics_merged['report'] + "\n")

                # Log counting metrics if available
                if 'mae_mse_report' in locals():
                    log.write(f"\nCounting Performance:\n{mae_mse_report}\n")

                log.write(f"\nEpoch time: {time_to_str(timer() - epoch_start, 'min')}\n")
                log.write("="*50 + "\n")



cross_val_lists = ['0', '1', '2', '3', '4']
for cross_val_index in cross_val_lists:
    log.write('\n\ncross_val_index: ' + cross_val_index + '\n\n')
    if True:
        trainval_test(cross_val_index, sigma=30 * 0.1, lam=6 * 0.1)




cross_val_index: 0

train      0 | 0.837 | 1.394 | 0.952 | 1.094 |  0 hr 02 min
train      1 | 0.761 | 1.164 | 0.800 | 0.934 |  0 hr 04 min
train      2 | 0.746 | 1.131 | 0.775 | 0.909 |  0 hr 06 min
train      3 | 0.737 | 1.123 | 0.768 | 0.901 |  0 hr 09 min
train      4 | 0.723 | 1.101 | 0.749 | 0.882 |  0 hr 11 min
train      5 | 0.708 | 1.089 | 0.738 | 0.869 |  0 hr 14 min
train      6 | 0.682 | 1.070 | 0.719 | 0.848 |  0 hr 16 min
train      7 | 0.669 | 1.054 | 0.702 | 0.833 |  0 hr 18 min
train      8 | 0.665 | 1.046 | 0.694 | 0.826 |  0 hr 21 min
train      9 | 0.660 | 1.025 | 0.677 | 0.811 |  0 hr 23 min
train     10 | 0.637 | 1.001 | 0.654 | 0.788 |  0 hr 26 min
train     11 | 0.624 | 0.987 | 0.640 | 0.774 |  0 hr 29 min
train     12 | 0.620 | 0.975 | 0.630 | 0.765 |  0 hr 32 min
train     13 | 0.613 | 0.968 | 0.624 | 0.758 |  0 hr 35 min
train     14 | 0.610 | 0.962 | 0.621 | 0.754 |  0 hr 38 min
train     15 | 0.604 | 0.954 | 0.613 | 0.747 |  0 hr 41 min
train     16 | 0.6